In [1]:
# ============================================
# TF-IDF Based Recommendation/Search System
# ============================================

# Import Libraries
import pandas as pd
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Anish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Anish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:

# ============================================
# STEP 1: CREATE DATASET
# ============================================

documents = [
    {
        "title": "Space Robots",
        "description": "A group of robots travel through space to save humanity from aliens."
    },
    {
        "title": "Healthy Living",
        "description": "A guide about healthy food, exercise, and maintaining a balanced lifestyle."
    },
    {
        "title": "Ocean Mystery",
        "description": "Scientists explore the deep ocean and discover mysterious creatures."
    },
    {
        "title": "Cooking Master",
        "description": "Learn delicious recipes, healthy meals, and cooking techniques."
    },
    {
        "title": "Galaxy Adventure",
        "description": "A young pilot begins an exciting adventure across distant galaxies."
    },
    {
        "title": "Fitness Journey",
        "description": "Tips for weight loss, healthy eating, and daily workout routines."
    },
    {
        "title": "AI Future",
        "description": "Artificial intelligence and robots transform modern society."
    },
    {
        "title": "Jungle Survival",
        "description": "A survival expert teaches how to survive in dangerous jungles."
    },
    {
        "title": "Space Mission",
        "description": "Astronauts go on a risky mission to Mars with intelligent machines."
    },
    {
        "title": "Food Science",
        "description": "Nutrition experts explain healthy diets and food science concepts."
    }
]

# Convert to DataFrame
df = pd.DataFrame(documents)

print("DATASET")
print(df)

DATASET
              title                                        description
0      Space Robots  A group of robots travel through space to save...
1    Healthy Living  A guide about healthy food, exercise, and main...
2     Ocean Mystery  Scientists explore the deep ocean and discover...
3    Cooking Master  Learn delicious recipes, healthy meals, and co...
4  Galaxy Adventure  A young pilot begins an exciting adventure acr...
5   Fitness Journey  Tips for weight loss, healthy eating, and dail...
6         AI Future  Artificial intelligence and robots transform m...
7   Jungle Survival  A survival expert teaches how to survive in da...
8     Space Mission  Astronauts go on a risky mission to Mars with ...
9      Food Science  Nutrition experts explain healthy diets and fo...


In [3]:
# ============================================
# STEP 2: TEXT PREPROCESSING
# ============================================

# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenization
    words = text.split()

    # Remove stopwords + Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    # Join words back
    return " ".join(words)

# Apply preprocessing
df['processed_text'] = (
    df['title'] + " " + df['description']
).apply(preprocess_text)

print("\nPROCESSED TEXT")
print(df[['title', 'processed_text']])



PROCESSED TEXT
              title                                     processed_text
0      Space Robots  space robot group robot travel space save huma...
1    Healthy Living  healthy living guide healthy food exercise mai...
2     Ocean Mystery  ocean mystery scientist explore deep ocean dis...
3    Cooking Master  cooking master learn delicious recipe healthy ...
4  Galaxy Adventure  galaxy adventure young pilot begin exciting ad...
5   Fitness Journey  fitness journey tip weight loss healthy eating...
6         AI Future  ai future artificial intelligence robot transf...
7   Jungle Survival  jungle survival survival expert teach survive ...
8     Space Mission  space mission astronaut go risky mission mar i...
9      Food Science  food science nutrition expert explain healthy ...


In [4]:
# ============================================
# STEP 3: TF-IDF VECTORIZATION
# ============================================

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df['processed_text'])

print("\nTF-IDF MATRIX SHAPE")
print(tfidf_matrix.shape)



TF-IDF MATRIX SHAPE
(10, 72)


In [5]:



# ============================================
# STEP 4: USER QUERY
# ============================================

user_query = "space adventure with robots"

# Preprocess query
processed_query = preprocess_text(user_query)

print("\nUSER QUERY")
print(user_query)

print("\nPROCESSED QUERY")
print(processed_query)

# Convert query to TF-IDF vector
query_vector = vectorizer.transform([processed_query])



USER QUERY
space adventure with robots

PROCESSED QUERY
space adventure robot


In [6]:

# ============================================
# STEP 5: COSINE SIMILARITY
# ============================================

similarity_scores = cosine_similarity(query_vector, tfidf_matrix)

# Convert scores to list
scores = similarity_scores[0]

# Add scores to dataframe
df['similarity_score'] = scores


In [7]:

# ============================================
# STEP 6: TOP 3 RECOMMENDATIONS
# ============================================

top_results = df.sort_values(
    by='similarity_score',
    ascending=False
).head(3)

print("\nTOP 3 RECOMMENDATIONS\n")

for index, row in top_results.iterrows():

    print("Title:", row['title'])
    print("Description:", row['description'])
    print("Similarity Score:", round(row['similarity_score'], 4))
    print("-" * 50)


TOP 3 RECOMMENDATIONS

Title: Space Robots
Description: A group of robots travel through space to save humanity from aliens.
Similarity Score: 0.563
--------------------------------------------------
Title: Galaxy Adventure
Description: A young pilot begins an exciting adventure across distant galaxies.
Similarity Score: 0.3418
--------------------------------------------------
Title: AI Future
Description: Artificial intelligence and robots transform modern society.
Similarity Score: 0.1663
--------------------------------------------------
